# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Etapas de Pré-processamento
Antes da análise, o notebook prepara as transcrições para que o conteúdo fique mais organizado e fácil de processar.

### O que é feito
1. **Limpeza básica do texto**
   - remove espaços extras no início e no fim de cada linha;
   - padroniza a formatação da transcrição.

2. **Divisão em blocos menores**
   - separa a transcrição em chunks de tamanho controlado;
   - mantém uma sobreposição entre blocos para preservar contexto.

3. **Identificação de referências bíblicas**
   - detecta quando há citação explícita de um livro, capítulo e versículo;
   - extrai essas referências em formato estruturado.

4. **Filtro de conteúdo relevante**
   - orienta o modelo a considerar apenas o que foi dito durante o Rosário;
   - ignora a reflexão final e outros trechos que não fazem parte da análise principal.

---

## Necessidades de Processamento
Para analisar corretamente as transcrições, o notebook precisa lidar com alguns tipos de conteúdo diferentes:

1. **Separar os momentos da fala**
   - identificar o ponto de divisão entre as reflexões do Terço e as reflexões do Dia;
   - separar os textos em duas partes: reflexão do Terço e reflexão do Dia.

2. **Filtrar trechos que não entram na análise principal**
   - remover as seções marcadas com a tag `[Música]`;
   - identificar e remover orações oficiais da Igreja Católica.

3. **Registrar músicas citadas durante a live**
   - extrair e documentar as músicas cantadas;
   - registrar o nome da música e o autor de cada uma.

4. **Contextualizar os ensinamentos dentro do Rosário**
   - considerar em que momento do Rosário cada ensinamento foi apresentado;
   - relacionar cada trecho ao terço e ao mistério meditado naquele instante.

---

## Recursos Externos Utilizados
Além do texto do notebook, este processo depende de alguns recursos que ficam fora dele, mesmo quando estão no mesmo projeto local:

1. **Vector Store da Bíblia**
   - armazenado em `../Bíblia VectorStore/biblia_vectorstore`;
   - usado para buscar passagens bíblicas semelhantes aos trechos analisados.

2. **Banco SQLite da Bíblia**
   - acessado em `../Bíblia VectorStore/biblia.db`;
   - contém os versículos consultados durante a extração das referências.

3. **Modelo de embeddings**
   - `sentence-transformers/all-MiniLM-L6-v2`;
   - transforma trechos de texto em vetores para permitir a busca por similaridade.

4. **Modelo de linguagem local**
   - executado via `Ollama`;
   - gera as respostas e faz a extração estruturada das informações.

5. **Arquivos de entrada e saída do projeto**
   - leitura em `../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text`;
   - saída prevista em `../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text`.


## 1. Configurações

### 1.1. Importação de Bibliotecas


In [1]:
import os
import logging
from contextlib import contextmanager

from dotenv import load_dotenv

from tqdm.notebook import tqdm

# from langchain_ollama import ChatOllama
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain.chat_models import init_chat_model
from langchain.tools import tool

import sys
from pathlib import Path

src_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "bible_vectorstore").exists()
    and (candidate / "rosarios_quaresma_frei_gilson").exists()
)
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from bible_vectorstore.bible_model import BibleExcerpt, BibleExcerpts  # noqa: E402
from rosarios_quaresma_frei_gilson.utils import (  # noqa: E402
    BinaryResponse,
    clean_tags,
    normalize_whitespace,
    trim_line_whitespace,
)

load_dotenv()

True

In [2]:
MODEL_PROVIDER = "ollama"
# MODEL_PROVIDER = "google_genai"
# MODEL_PROVIDER = "google_vertexai"

# MODEL = "Gemma-3-Gaia-PT-BR-4b-it-f16:latest"
# MODEL = "batiai/gemma4-e4b:q4"
MODEL = "gemma4:e2b-it-qat"

# List of Models: https://ai.google.dev/gemini-api/docs/models
# MODEL = "gemini-3.5-flash"

### 1.2. Hiperparâmetros

**Chunk size / Overlap**


Tabela: percentis da quantidade de caracteres por versículo bíblico

| Métrica | Valor Associado |
| :--- | :--- |
| **Percentil p0** | 2 |
| **Percentil p10** | 32 |
| **Percentil p25 (Q1)** | 65 |
| **Percentil p50 (Mediana)** | 94 |
| **Percentil p75 (Q3)** | 135 |
| **Percentil p90** | 176 |
| **Percentil p95** | 202 |
| **Percentil p99** | 256 |
| **Percentil p100** | 576 |


In [3]:
MIN_SIMILARITY_THRESHOLD = 0.65
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

## 2.0. Processamento de texto

In [4]:
system_message = """
    Você é um especialista na fé católica.  
    Receberá um **texto com a transcrição da oração do Santo Rosário**, rezado ao vivo pelo Frei Gilson durante um dia da Quaresma de 2025.

    A oração está dividida em duas partes:
    - A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    - A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    ⚠️ **IMPORTANTE:**  
    Ignore qualquer parte da transcrição que seja da parte da **reflexão**.  
    Use **apenas o que for dito durante a oração do Rosário**.

    ---

    ### Sua missão é extrair e organizar as informações nos itens de 1 a 5 abaixo:

    ---

    ## 1. Temática principal

    - Identifique a principal ideia que Frei Gilson ensina enquanto reza o Rosário.
    - Resuma em até 3 parágrafos.
    - Use somente ensinamentos que ele falou **durante a oração do Santo Rosário**.

    ---

    ## 2. Temáticas secundárias

    - Identifique de 2 a 5 temas que Frei Gilson também falou **durante a oração do Santo ROsário**.
    - Para cada tema:
    - Dê um título claro (ex: *Perseverança na fé*).
    - Escreva 1 a 2 parágrafos com os ensinamentos relacionados.

    ---

    ## 3. Versículos da Bíblia

    Vou lhe fornecer os **versículos bíblicos citados durante a oração**.
    Você só ignorará versículos que sejam referentes a orações como o Pai nosso, Ave Maria, Credo etc,
    mas manterá todos os demais versículos.
    Não se esqueça de nenhum versículo ou intervalo de versículos que eu lhe fornecer, a menos que citado antyeriormente nas excessões.
    Traga apenas versículos que foram citados durante a oração do Santo Rosário.
    Traga a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson.
    Para cada um:

    - Escreva assim:  
    `(Livro Capítulo, Versículo)`: <Transcrição integral do versículo> ou
    `(Livro Capítulo, Versículo-Início–Versículo-Fim)`: <Transcrição integral do intervalo de versículos>

    - Logo abaixo, escreva:
    **Ensinamentos:**: <Parágrafo ou frase que explique o que o Frei falou sobre esse versículo durante a oração>

    Se o versículo for relacionado a algum mistério do Santo Rosário (como `Segundo Mistério Doloroso` ou `Quinto Mistério Gozozo`), diga isso.

    ---

    ## 4. Músicas

    Se Frei Gilson mencionar músicas durante a oração:

    - Diga o nome da música e o artista, como neste exemplo:
    <Nome da música> - <Artista>: <contexto>  

    - Depois escreva o que Frei Gilson disse sobre a música.

    ---

    ## 5. Eventos de Agenda

    Se Frei Gilson mencionar **eventos** (missas, encontros, lives etc.):
    Traga todos os eventos citados durante a oração ou ao final do Santo Rosário.
    - Traga o nome do evento, a data e o local.
    - Depois escreva o que ele falou sobre esse evento.

    ---

    ## ⚠️ Regras finais:

    - NÃO copie orações conhecidas (Pai Nosso, Ave Maria, Credo etc.).
    - Não explique o rosário, exceto se dor necessário para compreender o respectivo ensinamento.
    - NÃO traga informações da reflexão final que ocorre após o rosário. Traga só o que é dito durante o Rosário.
    - NÃO invente nada. Só use o que estiver escrito.
    - NÃO deixe seções em branco. Se algo não aparecer, **remova a seção inteira**.
    ```
"""

## 3.0 Detecção de trechos da bíblia

In [5]:
# 0. Carregue a transcrição (demo)
# transcription = """
# Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
# Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
# """

# 1. Divida a transcrição
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
# chunks = splitter.create_documents([transcription])

# 2. Carregue os embeddings e a base vetorial
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma(
    collection_name="biblia",
    persist_directory="../Bíblia VectorStore/biblia_vectorstore",
    embedding_function=embedding_model,
)


# 3. Defina a enum e o parser
binary_parser = EnumOutputParser(enum=BinaryResponse)


# 4. Modelo Pydantic para saída estruturada de referência bíblica
# class BibleExcerpt(BaseModel):
#     book: BookEnum = Field(
#         ...,
#         description="Nome do livro da Bíblia explicitamente anunciado. Não deve conter número do capítulo.",
#         examples=ORDERED_BOOKS,
#     )
#     chapter: int = Field(
#         ...,
#         description="Número do capítulo explicitamente anunciado. Não deve conter número de versículos.",
#     )
#     verse_start: int = Field(
#         ...,
#         description="Número do primeiro versículo do intervalo explicitamente anunciado, se for um único versículo. Se for um intervalo, é o primeiro versículo do intervalo.",
#     )
#     verse_end: int = Field(
#         ...,
#         description="Número do último versículo do intervalo explicitamente anunciado, se for um intervalo. Se não houver intervalo, igual ao verse_start.",
#     )

#     @field_validator("verse_end", mode="before")
#     @classmethod
#     def set_verse_end(cls, v, values):
#         if v is None:
#             return info.data.get("verse_start")
#         return v


# class BibleExcerpts(BaseModel):
#     bible_excerpts: List[BibleExcerpt] = Field(
#         ..., description="Lista com todos os trechos da bíblia identificados"
#     )

#     def sort_and_deduplicate(self):
#         self.bible_excerpts.sort(
#             key=lambda x: (
#                 ORDERED_BOOKS.index(x.book),
#                 x.chapter,
#                 x.verse_start,
#             )
#         )
        
#         unique_excerpts = []
#         seen = set()
        
#         for excerpt in self.bible_excerpts:
#             identifier = (
#                 excerpt.book,
#                 excerpt.chapter,
#                 excerpt.verse_start,
#                 excerpt.verse_end,
#             )
#             if identifier not in seen:
#                 seen.add(identifier)
#                 unique_excerpts.append(excerpt)
                
#         self.bible_excerpts = unique_excerpts

#     def __str__(self):
#         return "\t".join(
#             f"{excerpt.book} "
#             f"{excerpt.chapter}:"
#             f"{excerpt.verse_start}-"
#             f"{excerpt.verse_end}"
#             for excerpt in self.bible_excerpts
#         )


# Parser para saída estruturada
# bible_reference_parser = PydanticOutputParser(pydantic_object=BibleExcerpts)
# structured_llm = llm.with_structured_output(BibleExcerpts)

# Novo prompt: pede saída JSON estruturada para referência bíblica
# bible_reference_prompt_template = """
# Você é um assistente que analisa trechos de transcrição de fala e identifica qual é a **passagem bíblica** citada de forma clara e explícita.

# Sua tarefa é identificar e **extrair** as seguintes informações, mesmo que existam variações de linguagem falada ou erros de transcrição:
# - O **nome do livro da Bíblia**
# - O **número do capítulo**
# - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

# Sua resposta deve ser exatamente neste formato **sem explicações, comentários ou qualquer outro texto**:
# {format_instructions}

# Texto a ser analisado:
# {query}
# """

# bible_reference_prompt = PromptTemplate(
#     template=bible_reference_prompt_template,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": bible_reference_parser.get_format_instructions()
#     },
# )

# print("Format Instructions:")
# print(bible_reference_parser.get_format_instructions())
# 5. LLM local
# Funciona bem com os modelos instruct:
# - llama3.1:8b-instruct-q5_K_M
# - mistral:7b-instruct


# 6. Cadeia completa manual (RAG customizado)


# def format_context(docs: List[Document], show_option=False):
#     if show_option:
#         return "\n\n".join(
#             f"Excerpt {i + 1}: {doc.metadata['book']} {doc.metadata['chapter']}:{doc.metadata['verse']} {doc.page_content}"
#             for (i, doc) in enumerate(docs)
#         )
#     return "\n\n".join(
#         f"{doc.metadata['book']} {doc.metadata['chapter']}:{doc.metadata['verse']} {doc.page_content}"
#         for doc in docs
#     )


@contextmanager
def silence_retriever_warning():
    logger = logging.getLogger("langchain_core.vectorstores")
    previous_level = logger.level

    try:
        logger.setLevel(logging.ERROR)
        yield
    finally:
        logger.setLevel(previous_level)


@tool
def retriever_bible_passages_tool(
    query: str, k: int = 3, min_similarity_threshold: float = MIN_SIMILARITY_THRESHOLD
):
    """
    Recupera as passagens bíblicas mais similares ao trecho fornecido.
    """
    retriever = db.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": k,
            "score_threshold": min_similarity_threshold,
        },
    )

    with silence_retriever_warning():
        return retriever.invoke(query)[:k]


# def retrieve_books_and_chapters(docs: List[Document]) -> str:
#     """
#     Retrieve all verses from the specific book and chapter found in docs.
#     Assumes a table 'versiculos' with columns: id, book, chapter, verse, text, line_number.
#     """
#     conn = sqlite3.connect("../Bíblia VectorStore/biblia.db")
#     cursor = conn.cursor()
#     results = []
#     seen = set()
#     for doc in docs:
#         book = doc.metadata.get("book")
#         chapter = doc.metadata.get("chapter")
#         if book and chapter and (book, chapter) not in seen:
#             seen.add((book, chapter))
#             results.append(f"\n\nLivro: {book} - Capítulo: {chapter}")
#             cursor.execute(
#                 "SELECT book, chapter, verse, text FROM versiculos WHERE book=? AND chapter=? ORDER BY verse ASC",
#                 (book, chapter),
#             )
#             for row in cursor.fetchall():
#                 results.append(f"{row[2]}: {row[3]}")
#     conn.close()
#     return "\n".join(results) if results else "Nenhum resultado encontrado."

In [6]:
# Pipeline binário para detecção de referência bíblica
# binary_bible_reference_template = """
# Você é um assistente que analisa trechos de transcrição de fala e identifica se há uma **citação bíblica clara e explícita**.

# Sua tarefa é:
# 1. Verificar se o trecho contém uma **anunciação clara** de que será citada uma **passagem bíblica**.
# 2. A citação deve conter, de forma explícita (mesmo com erros de transcrição ou variações da fala):
#    - O **nome do livro da Bíblia**
#    - O **número do capítulo**
#    - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

# Se você encontrar essa citação bíblica clara, responda apenas com "Sim.". Caso contrário, responda apenas com "Não.".
# Não escreva nenhuma explicação, justificativa ou comentário.
# O resultado deve ser apenas "Sim." ou "Não." — nada além disso.

# Texto a ser analisado:
# {query}
# """

# binary_bible_reference_prompt = PromptTemplate(
#     template=binary_bible_reference_template,
#     input_variables=["query"],
# )

# binary_bible_reference_chain = (
#     {"query": lambda x: x["query"]}
#     | binary_bible_reference_prompt
#     | llm
#     | binary_parser
# )
# binary_bible_reference_tool = binary_bible_reference_chain.as_tool()

In [7]:
# Se der problema com Ollama use isto
# llm = ChatOllama(model=MODEL)

llm = init_chat_model(
    MODEL,
    model_provider=MODEL_PROVIDER,
)

In [8]:
# fixing_parser = OutputFixingParser.from_llm(parser=bible_reference_parser, llm=llm)

# bible_reference_chain = {"query": lambda x: x["query"]} | bible_reference_prompt | llm

structured_llm = llm.with_structured_output(BibleExcerpts, include_raw=False)
llm_with_tools = llm.bind_tools([retriever_bible_passages_tool])


BIBLE_EXCERPTS_JOIN_PROMPT = """
Retorne os trechos da bíblia abaixo, juntando se forem contínuos no mesmo capítulo do mesmo livro:

Trechos:
{bible_excerpts}
"""


EXTRACT_BIBLE_PASSAGES_SYSTEM_PROMPT = """
Você é um teólogo renomado e conhecido por conseguir identificar trechos da bíblia.

Você deve identificar e isolar trechos nos textos a seguir que possam conter versículos da bíblia, um candidato a versículo por trecho.
Envie cada trecho candodato a versículo isoladamente, individualmente, para a ferramenta `retriever_bible_passages_tool` e colete a resposta.

Para cada trecho que for um possível versículo, você deve atentar-se para o início e o fim.
Os trechos da bíblia que serão lidos são sempre anúnciados previamente. Use essa anunciação para ajudar a determinar início de trecho de bíblia.
O fim do trecho de bíblia geralmente pode ser inferido pela mudança de palavriado bíblico; pela mudança na narrativa;
pela mudança dos verbos habituais e conjugações da bíblia para conjugações mais normais; pelo início de uma explicação; entre outras formas.

Envie à ferramenta cada um dos trechos contíguos que há possibilidade de conter um versículo contíguo da bíblia.
"""


def find_bible_versicles(text: str, verbose=False) -> BibleExcerpts:
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    if verbose:
        print("[generate_query_or_respond] invoking: ", text)

    messages = [
        {"role": "system", "content": EXTRACT_BIBLE_PASSAGES_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": text,
        },
    ]

    bible_excerpts: BibleExcerpts = BibleExcerpts(bible_excerpts=[])

    ai_msg = llm_with_tools.invoke(messages)
    if not ai_msg.tool_calls:
        return bible_excerpts

    tool_messages = []
    for tc in ai_msg.tool_calls:
        if verbose:
            print("\nTool call: ", tc)
        tool_result = retriever_bible_passages_tool.invoke(tc["args"]["query"])
        if verbose:
            print("\nTool call response: ", tool_result)
        tool_messages.append(
            {
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": str(tool_result),
            }
        )

    final_prompt = f"""
    Verifique se os resultados recuperados correspondem realmente
    ao trecho original.

    Trecho original:
    {text}

    Resultados do retriever:
    {tool_messages}

    Retorne somente as referências bíblicas que realmente correspondem
    ao trecho original. Ignore resultados apenas semanticamente parecidos.
    """

    raw_refs = structured_llm.invoke(trim_line_whitespace(final_prompt))

    if isinstance(raw_refs, BibleExcerpts):
        refs = raw_refs
    else:
        refs = BibleExcerpts.model_validate(raw_refs)

    if verbose:
        print("\nBible excerpts:\n", str(refs))

    bible_excerpts.bible_excerpts.extend(refs.bible_excerpts)

    return bible_excerpts


def get_all_bible_passages(transcription: str, verbose: bool = False) -> BibleExcerpts:
    """
    Extrai referências bíblicas de uma transcrição, com prints opcionais para depuração.
    """
    bible_passages: BibleExcerpts = BibleExcerpts(bible_excerpts=[])
    chunks = splitter.create_documents([transcription])
    if verbose:
        print(f"[get_bible_passages] Total de chunks: {len(chunks)}")

    for idx, chunk in enumerate(tqdm(chunks, desc="Processando chunks")):
        if verbose:
            print(f"\n[get_bible_passages] Chunk {idx + 1}/{len(chunks)}:")
            print(f"\nConteúdo do chunk:\n\n{chunk.page_content}\n")

        response = find_bible_versicles(chunk.page_content, verbose=verbose)
        if verbose:
            print("\nLLM response:")
            print(response)

        bible_passages.bible_excerpts.extend(response.bible_excerpts)

    bible_passages.sort_and_deduplicate()
    return bible_passages


# Exemplo de uso:
# Propositalmente, chamamos o livro errado.
# Livro correto: 1 Corintios
# bible_refs = get_all_bible_passages(
#     transcription=trim_line_whitespace(
#         """
#         Livro de Primeiro João, capítulo 3, versículos 16 a 17.

#         Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
#         Se alguém destruir o Templo de Deus, Deus o destruirá.
#         Porque o templo de Deus é sagrado – e isso sois vós.
#         """
#     ),
#     verbose=True,
# )
# print("Referências bíblicas extraídas:")
# for ref in bible_refs:
#     print(ref)

## 4.0. Leitura dos Arquivos

In [9]:
# model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)


CONTENT_AND_BIBLE_REFS_TEMPLATE = """
# Conteúdo do Santo Rosário
\n{content}
\n# Referências bíblicas extraídas
\n{bible_refs}
"""

for i in tqdm(range(13, 14), desc="Processando arquivos"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                clean_content = clean_tags(conteudo)
                clean_content = normalize_whitespace(clean_content)
                clean_content = trim_line_whitespace(clean_content)

                print(f"Arquivo: {titulo_source}")

                bible_refs = get_all_bible_passages(clean_content, verbose=False)
                print("\nReferências bíblicas extraídas:")

                print("\nBible references found:\n")
                print(bible_refs)

                # Temp print to check the content and references before writing to file
                print("CONTENT_AND_BIBLE_REFS_TEMPLATE:\n")
                print(
                    CONTENT_AND_BIBLE_REFS_TEMPLATE.format(
                        content=clean_content, bible_refs=str(bible_refs)
                    )
                )

                messages = [
                    {"role": "system", "content": trim_line_whitespace(system_message)},
                    {
                        "role": "user",
                        "content": CONTENT_AND_BIBLE_REFS_TEMPLATE.format(
                            content=clean_content, bible_refs=str(bible_refs)
                        ),
                    },
                ]

                # Live stream final response for the document
                chunks = []
                for text in llm.stream(messages):
                    chunks.append(text.text())
                    print(text.text(), end="", flush=True)
                response = "".join(chunks)

                with open(
                    f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                ) as f:
                    f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")

    break

Diretório atual: /home/bruno/Workspace/MariaGPT/src/rosarios_quaresma_frei_gilson


Processando arquivos:   0%|          | 0/1 [00:00<?, ?it/s]

Arquivo: Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo.txt


Processando chunks:   0%|          | 0/276 [00:00<?, ?it/s]

KeyboardInterrupt: 